# Clinical Behavioral Testing (Colab)
This notebook installs dependencies, creates sample data, trains a small classifier, evaluates it, and runs behavioral tests (minimal pairs). It is self-contained and does not require cloning the repo.

In [ ]:
%pip -q install transformers==4.43.3 datasets==2.20.0 evaluate==0.4.2 scikit-learn==1.4.2 torch==2.3.1 pandas==2.2.2 numpy==1.26.4 tqdm==4.66.4 pyyaml==6.0.2 accelerate==0.33.0 sentencepiece==0.2.0 protobuf==5.27.0

In [ ]:
import os, json, textwrap, random
from pathlib import Path
random.seed(42)
base = Path('/content')
(base/'data'/'sample').mkdir(parents=True, exist_ok=True)
(base/'behavioral_tests').mkdir(parents=True, exist_ok=True)
train = [
 {"text": "The patient has pneumonia.", "label": 1},
 {"text": "No evidence of pneumonia.", "label": 0},
 {"text": "Diabetes is present.", "label": 1},
 {"text": "No diabetes is present.", "label": 0},
 {"text": "BP elevated at 160/100.", "label": 1},
 {"text": "Blood pressure is within normal range.", "label": 0},
 {"text": "There is evidence of hypertrophy.", "label": 1},
 {"text": "There is no evidence of hypertrophy.", "label": 0},
 {"text": "CXR shows consolidation.", "label": 1},
 {"text": "Chest x-ray is clear.", "label": 0}
]
valid = [
 {"text": "The patient has hypertension.", "label": 1},
 {"text": "No hypertension is documented.", "label": 0},
 {"text": "CXR indicates consolidation.", "label": 1},
 {"text": "Chest x-ray is normal.", "label": 0},
 {"text": "There is evidence of infection.", "label": 1},
 {"text": "There is no evidence of infection.", "label": 0}
]
test = [
 {"text": "The patient has pneumonia.", "label": 1},
 {"text": "No evidence of pneumonia.", "label": 0},
 {"text": "The chest x-ray is normal.", "label": 0},
 {"text": "CXR shows consolidation.", "label": 1},
 {"text": "There is no evidence of infection.", "label": 0},
 {"text": "There is evidence of infection.", "label": 1}
]
with open(base/'data'/'sample'/'train.jsonl','w') as f:
  for r in train: f.write(json.dumps(r)+'
')
with open(base/'data'/'sample'/'valid.jsonl','w') as f:
  for r in valid: f.write(json.dumps(r)+'
')
with open(base/'data'/'sample'/'test.jsonl','w') as f:
  for r in test: f.write(json.dumps(r)+'
')
tests = [
 {"text_a": "The patient has pneumonia.", "text_b": "The patient does not have pneumonia.", "expected_relation": "flip"},
 {"text_a": "Diabetes is present.", "text_b": "No diabetes is present.", "expected_relation": "flip"},
 {"text_a": "The CXR is normal.", "text_b": "The chest x-ray is normal.", "expected_relation": "same"}
]
with open(base/'behavioral_tests'/'tests.jsonl','w') as f:
  for r in tests: f.write(json.dumps(r)+'
')
print('Data ready under /content/data and /content/behavioral_tests')


In [ ]:
import numpy as np
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, set_seed
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import json, os
set_seed(42)
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  preds = np.argmax(logits, axis=-1)
  avg = 'binary' if len(set(labels))==2 else 'macro'
  return {
    'accuracy': float(accuracy_score(labels, preds)),
    'precision': float(precision_score(labels, preds, average=avg, zero_division=0)),
    'recall': float(recall_score(labels, preds, average=avg, zero_division=0)),
    'f1': float(f1_score(labels, preds, average=avg, zero_division=0)),
  }
train_file = '/content/data/sample/train.jsonl'
valid_file = '/content/data/sample/valid.jsonl'
output_dir = '/content/outputs/bert-baseline'
raw = load_dataset('json', data_files={'train': train_file, 'validation': valid_file})
# normalize labels to 0..N-1
labels = sorted(list({int(x) for x in raw['train']['label']}))
label2id = {str(v): i for i, v in enumerate(labels)}
id2label = {i: str(v) for i, v in enumerate(labels)}
def normalize_labels(example):
  example['label'] = label2id[str(int(example['label']))]
  return example
raw = raw.map(normalize_labels)
tok = AutoTokenizer.from_pretrained('bert-base-uncased', use_fast=True)
def tok_fn(batch):
  return tok(batch['text'], truncation=True, max_length=256)
tokenized = raw.map(tok_fn, batched=True, remove_columns=[c for c in raw['train'].column_names if c not in ['label']])
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(label2id), id2label=id2label, label2id={k:int(v) for k,v in label2id.items()})
args = TrainingArguments(output_dir=output_dir, num_train_epochs=3, per_device_train_batch_size=16, per_device_eval_batch_size=32, learning_rate=2e-5, evaluation_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True, metric_for_best_model='f1', report_to=[])
trainer = Trainer(model=model, args=args, train_dataset=tokenized['train'], eval_dataset=tokenized['validation'], tokenizer=tok, compute_metrics=compute_metrics)
trainer.train()
os.makedirs(output_dir, exist_ok=True)
trainer.save_model(output_dir)
tok.save_pretrained(output_dir)
with open(os.path.join(output_dir,'label_map.json'),'w') as f:
  json.dump({'label2id': label2id, 'id2label': {str(k): v for k, v in id2label.items()}}, f, indent=2)
print('Training done. Model at', output_dir)


In [ ]:
from datasets import load_dataset
from sklearn.metrics import classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch, numpy as np, json
test_file = '/content/data/sample/test.jsonl'
model_dir = '/content/outputs/bert-baseline'
raw_test = load_dataset('json', data_files={'test': test_file})
tok = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
def tokenize(batch):
  return tok(batch['text'], truncation=True, max_length=256)
tok_test = raw_test.map(tokenize, batched=True)
enc = tok_test['test'].with_format('torch')
loader = torch.utils.data.DataLoader(enc, batch_size=32)
preds, labels = [], []
model.eval()
with torch.no_grad():
  for batch in loader:
    inputs = {k: v for k,v in batch.items() if k in ['input_ids','attention_mask','token_type_ids']}
    logits = model(**inputs).logits
    preds.extend(logits.argmax(dim=-1).cpu().tolist())
    labels.extend(batch['label'].cpu().tolist())
print(classification_report(labels, preds, digits=4))


In [ ]:
import json, torch, os
from transformers import AutoTokenizer, AutoModelForSequenceClassification
def softmax(x):
  e_x = torch.exp(x - x.max(dim=-1, keepdim=True).values)
  return e_x / e_x.sum(dim=-1, keepdim=True)
def predict(model, tokenizer, texts, max_length=256, device='cpu'):
  enc = tokenizer(texts, truncation=True, max_length=max_length, return_tensors='pt', padding=True)
  with torch.no_grad():
    logits = model(**enc).logits
    probs = softmax(logits).cpu().tolist()
    preds = logits.argmax(dim=-1).cpu().tolist()
  return preds, probs
tests_path = '/content/behavioral_tests/tests.jsonl'
tests = [json.loads(l) for l in open(tests_path)]
tok = AutoTokenizer.from_pretrained('/content/outputs/bert-baseline')
model = AutoModelForSequenceClassification.from_pretrained('/content/outputs/bert-baseline')
results, correct = [], 0
for t in tests:
  a, b = t['text_a'], t['text_b']
  expected = t.get('expected_relation','flip')
  preds, probs = predict(model, tok, [a,b])
  pa, pb = preds
  ok = (pa!=pb) if expected=='flip' else (pa==pb)
  correct += int(ok)
  results.append({**t, 'pred_a': int(pa), 'pred_b': int(pb), 'relation_ok': bool(ok), 'probs_a': probs[0], 'probs_b': probs[1]})
summary = {'total_pairs': len(tests), 'relation_accuracy': correct/max(1,len(tests))}
os.makedirs('/content/reports', exist_ok=True)
with open('/content/reports/behavioral_report.json','w') as f:
  json.dump({'summary': summary, 'results': results}, f, indent=2)
summary
